In [1]:
!pip install yfinance ta xgboost lightgbm ngboost

     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     ----------------------------- -------- 41.0/52.8 kB 991.0 kB/s eta 0:00:01
     -------------------------------------- 52.8/52.8 kB 906.2 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --- ------------------------------------ 0.1/1.5 MB 3.5 MB/s eta 0:00:01
   ------- -------------------------------- 0.3/1.5 MB 2.8 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------  1.4/1.5 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 8.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
   ------------

In [5]:
pip install --upgrade yfinance

   ---------------------------------------- 0.0/130.8 kB ? eta -:--:--
   ------------------ --------------------- 61.4/130.8 kB 1.1 MB/s eta 0:00:01
   ------------------------------------- -- 122.9/130.8 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 130.8/130.8 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   --- ------------------------------------ 0.2/1.7 MB 4.5 MB/s eta 0:00:01
   ------ --------------------------------- 0.3/1.7 MB 3.4 MB/s eta 0:00:01
   ------------- -------------------------- 0.6/1.7 MB 5.1 MB/s eta 0:00:01
   --------------------------- ------------ 1.2/1.7 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 8.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/182.8 kB ? eta -:--:--
   --------------------------------------- 182.8/182.8 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling

  You can safely remove it manually.
  You can safely remove it manually.


In [11]:
import pandas as pd
import numpy as np
import requests
import ta

def load_binance():
    url = "https://api.binance.com/api/v3/klines"
    params = {"symbol": "BTCUSDT","interval": "1d","limit": 1200}
    data = requests.get(url, params=params).json()

    df = pd.DataFrame(data, columns=[
        "time","open","high","low","close","volume",
        "_","_","_","_","_","_"
    ])

    df['Close'] = df['close'].astype(float)
    df['Volume'] = df['volume'].astype(float)

    return df[['Close','Volume']]

df = load_binance()

# ===== Feature chung =====
df['return'] = df['Close'].pct_change()
df['lag1'] = df['return'].shift(1)
df['lag2'] = df['return'].shift(2)
df['volatility'] = df['return'].rolling(14).std()

df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
macd = ta.trend.MACD(df['Close'])
df['macd'] = macd.macd()
df['macd_diff'] = macd.macd_diff()

df.dropna(inplace=True)

In [14]:
#LAB 1 – Quantile Regression
# LAB 1 – Quantile Regression (không dùng lightgbm)

from sklearn.linear_model import QuantileRegressor
import numpy as np

# ===== Target =====
df['target'] = df['return'].shift(-1)
df = df.dropna()

# ===== Feature =====
X = df[['lag1','lag2','volatility','rsi','macd','macd_diff']]
y = df['target']

# ===== Train/Test split =====
split = int(len(df)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ===== Train quantile models =====
q05 = QuantileRegressor(quantile=0.05, alpha=0).fit(X_train, y_train)
q50 = QuantileRegressor(quantile=0.5, alpha=0).fit(X_train, y_train)
q95 = QuantileRegressor(quantile=0.95, alpha=0).fit(X_train, y_train)

# ===== Predict =====
pred_05 = q05.predict(X_test)
pred_50 = q50.predict(X_test)
pred_95 = q95.predict(X_test)

# ===== Metrics =====

# Expected return
expected_return = pred_50

# Downside risk
downside_risk = pred_05

# Probability return > 0 (approx)
prob_win = np.mean(pred_50 > 0)

# ===== Simple strategy =====
# Buy khi median > 0 và risk thấp
signal = (pred_50 > 0) & (pred_05 > -0.02)

strategy_return = signal * y_test.values

# ===== Profit factor =====
profit = strategy_return[strategy_return > 0].sum()
loss = -strategy_return[strategy_return < 0].sum()

profit_factor = profit / loss if loss != 0 else 0

print("Expected return sample:", expected_return[:5])
print("Downside risk sample:", downside_risk[:5])
print("Probability win:", prob_win)
print("Profit factor:", profit_factor)

Expected return sample: [-0.00110716 -0.00045962 -0.00063429 -0.00028521  0.00028983]
Downside risk sample: [-0.02070253 -0.02460371 -0.02574335 -0.02603873 -0.03124535]
Probability win: 0.8247422680412371
Profit factor: 0


In [15]:
#lab2

In [27]:
from xgboost import XGBClassifier
from sklearn.metrics import precision_score

df['future'] = df['Close'].pct_change(5).shift(-5)
df['label'] = (df['future'] > df['volatility']).astype(int)

df2 = df.dropna()

X = df2[['volatility','Volume','rsi','macd_diff']]
y = df2['label']

model = XGBClassifier()
model.fit(X,y)

pred = model.predict(X)

# Precision breakout
precision = precision_score(y, pred)

# Strategy PnL
pnl = pred * df2['future']
total_pnl = pnl.sum()

print("Precision:", precision)
print("PnL:", total_pnl)

Precision: 1.0
PnL: 20.20423119510194


In [17]:
#lab3:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

conditions = [
    df['return'] > 0.02,
    df['return'].abs() < 0.01,
    df['return'] < -0.02
]

df['regime'] = np.select(conditions,[0,1,2],default=3)

df3 = df.dropna()

X = df3[['volatility','rsi','macd']]
y = df3['regime']

model = RandomForestClassifier()
model.fit(X,y)

print("Regime model done")


Regime model done


In [19]:
#lab4:
from sklearn.ensemble import GradientBoostingRegressor

df['proxy'] = df['return'].rolling(5).mean()
df['target'] = df['return'].shift(-1)

df4 = df.dropna()

X = df4[['proxy','volatility','rsi']]
y = df4['target']

model = GradientBoostingRegressor()
model.fit(X,y)

print("Cross asset done")

Cross asset done


In [20]:
#lab5:
from xgboost import XGBClassifier

df['direction'] = (df['return'].shift(-1) > 0).astype(int)

df5 = df.dropna()

X = df5[['volatility','rsi','macd']]
y = df5['direction']

model = XGBClassifier()
model.fit(X,y)

proba = model.predict_proba(X)[:,1]

print("Prob sample:", proba[:5])

Prob sample: [0.9404751  0.9404751  0.14487258 0.14851877 0.2744178 ]


In [21]:
#lab6:
from sklearn.ensemble import RandomForestClassifier

df['strength'] = np.where(df['return']>0.02,2,
                  np.where(df['return']>0.005,1,0))

df6 = df.dropna()

X = df6[['Volume','volatility','rsi']]
y = df6['strength']

model = RandomForestClassifier()
model.fit(X,y)

print("Breakout strength done")

Breakout strength done


In [22]:
#lab7:
from sklearn.ensemble import RandomForestClassifier

df['vol_regime'] = pd.qcut(df['volatility'],3,labels=[0,1,2])

df7 = df.dropna()

X = df7[['rsi','macd']]
y = df7['vol_regime']

model = RandomForestClassifier()
model.fit(X,y)

print("Vol regime done")


Vol regime done


In [24]:
#lab8
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression

df['target'] = df['return'].shift(-1)
df['direction'] = (df['target'] > 0).astype(int)

df8 = df.dropna()

X = df8[['volatility','rsi','macd']]

# Regression
model_reg = MLPRegressor(max_iter=500)
model_reg.fit(X, df8['target'])

# Classification
model_cls = LogisticRegression()
model_cls.fit(X, df8['direction'])

print("Hybrid done")

Hybrid done


In [25]:
#lab9:
from sklearn.linear_model import QuantileRegressor

window = 200
preds = []

for i in range(window, len(df)):
    train = df.iloc[i-window:i]
    test = df.iloc[i:i+1]

    X_train = train[['volatility','rsi','macd']]
    y_train = train['return']

    model = QuantileRegressor(quantile=0.5, alpha=0)
    model.fit(X_train, y_train)

    pred = model.predict(test[['volatility','rsi','macd']])
    preds.append(pred[0])

print("Walk-forward done:", len(preds))

Walk-forward done: 766
